In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

In [2]:
import os

dataset_path = "../../datasets/food-101/images"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

print("Dataset:", dataset_path)

Dataset: ../../datasets/food-101/images


In [3]:
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

validation_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Found 101000 files belonging to 101 classes.
Using 80800 files for training.
Found 101000 files belonging to 101 classes.
Using 20200 files for validation.


In [4]:
NUM_CLASSES = len(train_dataset.class_names)

print("Classes :", NUM_CLASSES)
print(train_dataset.class_names[:10])

Classes : 101
['apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare', 'beet_salad', 'beignets', 'bibimbap', 'bread_pudding', 'breakfast_burrito']


In [5]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_dataset = train_dataset.map(
    lambda x, y: (normalization_layer(x), y)
)

validation_dataset = validation_dataset.map(
    lambda x, y: (normalization_layer(x), y)
)

print("Normalization Completed")

Normalization Completed


In [6]:
data_augmentation = tf.keras.Sequential([

    layers.RandomFlip("horizontal"),

    layers.RandomRotation(0.2),

    layers.RandomZoom(0.2),

])

In [7]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

print("MobileNetV2 Loaded Successfully!")

MobileNetV2 Loaded Successfully!


In [8]:
base_model.trainable = False

print("Trainable:", base_model.trainable)

Trainable: False


In [9]:
model = models.Sequential([

    data_augmentation,

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dropout(0.3),

    layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )

])

In [10]:
model.build((None, 224, 224, 3))

In [11]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 101)            │       129,381 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,387,365 (9.11 MB)

 Trainable params: 129,381 (505.39 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [12]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model Compiled Successfully!")

Model Compiled Successfully!


In [13]:
EPOCHS = 10

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS 
)

Epoch 1/10
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 2304s 908ms/step - accuracy: 0.3609 - loss: 2.6997 - val_accuracy: 0.4910 - val_loss: 2.0283
Epoch 2/10
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 2085s 825ms/step - accuracy: 0.4321 - loss: 2.3638 - val_accuracy: 0.5091 - val_loss: 1.9628
Epoch 3/10
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 2348s 930ms/step - accuracy: 0.4442 - loss: 2.3282 - val_accuracy: 0.5187 - val_loss: 1.9360
Epoch 4/10
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 2154s 853ms/step - accuracy: 0.4491 - loss: 2.3031 - val_accuracy: 0.5171 - val_loss: 1.9384
Epoch 5/10
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 2028s 803ms/step - accuracy: 0.4497 - loss: 2.3021 - val_accuracy: 0.5213 - val_loss: 1.9363
Epoch 6/10
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 2008s 795ms/step - accuracy: 0.4518 - loss: 2.3049 - val_accuracy: 0.5168 - val_loss: 1.9634
Epoch 7/10
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 2308s 914ms/step - accuracy: 0.4513 - loss: 2.2956 - val_accuracy: 0.5252 - val_loss: 1.9130
Epoch 8/10
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 1972s 781ms/s

In [14]:
history.history.keys()

dict_keys(['accuracy', 'loss', 'val_accuracy', 'val_loss'])

In [15]:

# Create saved_models folder if it doesn't exist
os.makedirs("../saved_models", exist_ok=True)

# Save the trained model
model.save("../saved_models/mobilenetv2_food_model.keras")

print("✅ Model saved successfully!")

✅ Model saved successfully!


In [16]:
import pickle

with open("../saved_models/history.pkl", "wb") as f:
    pickle.dump(history.history, f)

print("✅ Training history saved successfully!")

✅ Training history saved successfully!


In [18]:
import tensorflow as tf

model = tf.keras.models.load_model("../saved_models/mobilenetv2_food_model.keras")

print("Model Loaded Successfully")

Model Loaded Successfully


In [17]:
import pickle

with open("../saved_models/history.pkl", "rb") as f:
    history = pickle.load(f)

print(history.keys())

dict_keys(['accuracy', 'loss', 'val_accuracy', 'val_loss'])
